In [30]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [31]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



In [32]:
simulator = BasicSimulator()

def quantum_random_bit():
    """Return a single truly-random bit by preparing |+> and measuring it."""
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    result = simulator.run(transpile(qc, simulator), shots=1).result()
    return int(list(result.get_counts().keys())[0])

# Choose number of qubits to send.
N = 40

In [33]:
# Alice
# Generate two random bit-strings of length N using the quantum RNG.
# alice_bits[i]  = the bit Alice wants to send for qubit i
# alice_bases[i] = 0 (standard) or 1 (diagonal) for qubit i

alice_bits = []
alice_bases = []
for i in range(N):
    alice_bits.append(quantum_random_bit())
    alice_bases.append(quantum_random_bit())

print("Alice's bits :", alice_bits)
print("Alice's bases:", alice_bases, "  (0=standard, 1=diagonal)")

Alice's bits : [0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1]
Alice's bases: [1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1]   (0=standard, 1=diagonal)


In [34]:
# prepare and send the qubits
qubits = []
for i in range(N):
    qc = QuantumCircuit(1, 1)
    # If the bit is 1, flip |0> to |1>.
    if alice_bits[i] == 1:
        qc.x(0)
    # If the basis is diagonal, rotate into it: H|0>=|+>, H|1>=|->.
    if alice_bases[i] == 1:
        qc.h(0)
    qubits.append(qc)

print(f"Alice has prepared {len(qubits)} qubits and sent them to Bob.")

Alice has prepared 40 qubits and sent them to Bob.


In [35]:
# BOB code
# Pick a random basis for each qubit Bob receives.
bob_bases = []
for i in range(N):
    bob_bases.append(quantum_random_bit())

print("Bob's bases:", bob_bases)

Bob's bases: [0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1]


In [36]:

# BOB — measure the qubits

bob_bits = []
for i in range(N):
    qc = qubits[i].copy()
    # Diagonal measurement = H then standard measurement.
    if bob_bases[i] == 1:
        qc.h(0)
    qc.measure(0, 0)
    result = simulator.run(transpile(qc, simulator), shots=1).result()
    outcome = int(list(result.get_counts().keys())[0])
    bob_bits.append(outcome)

print("Bob's bits :", bob_bits)

Bob's bits : [1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1]


In [37]:
#compare bases and sift
alice_key = []
bob_key = []
for i in range(N):
    if alice_bases[i] == bob_bases[i]:
        alice_key.append(alice_bits[i])
        bob_key.append(bob_bits[i])

print(f"Sifted key length: {len(alice_key)} out of {N} qubits sent.")
print("Alice's sifted key:", alice_key)
print("Bob's   sifted key:", bob_key)

Sifted key length: 22 out of 40 qubits sent.
Alice's sifted key: [0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1]
Bob's   sifted key: [0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1]


In [38]:
# RESULT

disagreements = 0
for i in range(len(alice_key)):
    if alice_key[i] != bob_key[i]:
        disagreements += 1

print(f"Disagreements between Alice and Bob in sifted key: {disagreements}")
print(f"Keys identical? {alice_key == bob_key}")

if alice_key == bob_key:
    print(f"\nShared secret key of length {len(alice_key)} successfully established.")

Disagreements between Alice and Bob in sifted key: 0
Keys identical? True

Shared secret key of length 22 successfully established.
